# 📊 Credit Risk & Loan Default Prediction: 01 - Exploratory Data Analysis (EDA)

### Objective:
Perform comprehensive exploratory data analysis on the Home Credit Default Risk application dataset. Analyze class imbalance, distributions, missing values, and key financial ratios correlated with default risk.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
%matplotlib inline


## 1. Load Raw Application Data


In [ ]:
df = pd.read_csv('../data/raw/application_train.csv')
print(f'Application Dataset Shape: {df.shape}')
df.head()


## 2. Target Distribution & Class Imbalance
In credit lending, defaults (positive class `1`) are typically rare (~8% of total loans). Addressing this imbalance is key for evaluation and thresholding.


In [ ]:
target_counts = df['TARGET'].value_counts()
target_perc = df['TARGET'].value_counts(normalize=True) * 100

print(pd.DataFrame({'Count': target_counts, 'Percentage (%)': target_perc.round(2)}))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='TARGET', palette=['#2563eb', '#dc2626'])
plt.title('Target Distribution (0 = Repaid, 1 = Default)')
plt.xlabel('Loan Default Status')
plt.ylabel('Count')
plt.show()


## 3. Missing Value Profiling


In [ ]:
missing = pd.DataFrame({
    'total_missing': df.isnull().sum(),
    'perc_missing': (df.isnull().sum() / len(df)) * 100,
    'data_type': df.dtypes
})
missing_cols = missing[missing['perc_missing'] > 0].sort_values(by='perc_missing', ascending=False)
print(f'Columns with missing values: {len(missing_cols)}')
missing_cols.head(25)


## 4. Key Financial & Demographic Features


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Credit Amount
sns.histplot(df['AMT_CREDIT'], bins=40, kde=True, ax=axes[0, 0], color='#2563eb')
axes[0, 0].set_title('Loan Credit Amount Distribution')

# Income (Capped at 99th percentile for clean visualization)
income_capped = df['AMT_INCOME_TOTAL'].clip(upper=df['AMT_INCOME_TOTAL'].quantile(0.99))
sns.histplot(income_capped, bins=40, kde=True, ax=axes[0, 1], color='#059669')
axes[0, 1].set_title('Income Distribution (99th Percentile Capped)')

# Contract Type vs Default
sns.countplot(data=df, x='NAME_CONTRACT_TYPE', hue='TARGET', ax=axes[1, 0], palette=['#2563eb', '#dc2626'])
axes[1, 0].set_title('Contract Type vs Default')

# Education vs Default
sns.countplot(data=df, y='NAME_EDUCATION_TYPE', hue='TARGET', ax=axes[1, 1], palette=['#2563eb', '#dc2626'])
axes[1, 1].set_title('Education Level vs Default')

plt.tight_layout()
plt.show()


## 5. External Credit Bureau Normalized Scores (`EXT_SOURCE_1/2/3`)


In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'TARGET']
corr = df[ext_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.3f', cbar=True)
plt.title('Correlation of External Sources with Target')
plt.show()
